# Enterprise AI Lab
## LangChain · Azure OpenAI · MLflow · Databricks RAG
### Step by Step Lab Guide

This lab guides you through building a complete **production ready AI pipeline on Azure**.  
Follow each section **in order** - every step builds on the previous one.

| # | Section | What You Build |
|---|---------|----------------|
| 0 | Environment Setup | Install packages, configure Azure credentials |
| 1 | Azure OpenAI Endpoints | Configure and test `AzureChatOpenAI` |
| 2 | LangChain LCEL Chains | Prompt → LLM → Parser pipelines |
| 3 | Sequential Chains | Multi-step document processing pipelines |
| 4 | Router Chains | Domain-based query routing |
| 5 | Memory & Stateful Conversations | Multi-turn chat with history |
| 6 | Build a Vector Store | FAISS index from internal documents |
| 7 | RAG Chain | Retrieval-Augmented Generation with Azure |
| 8 | MLflow Tracking | Log runs, metrics, register models |
| 9 | Databricks RAG | Production RAG on Databricks |
| 10 | Agents & Tools | LLM-driven tool-calling workflows |


## Prerequisites

Before starting, make sure you have:

### Azure Resources
- Azure subscription with Contributor role
- **Azure OpenAI resource** deployed (East US or West Europe recommended)
- Two model deployments in Azure OpenAI Studio:
  - `gpt-4o` — chat completion model
  - `text-embedding-ada-002` — embeddings model
-  Azure Databricks workspace

### Credentials You Need
| Variable | Where to Find It |
|----------|------------------|
| `AZURE_OPENAI_API_KEY` | Azure Portal → OpenAI resource → Keys and Endpoint |
| `AZURE_OPENAI_ENDPOINT` | Azure Portal → OpenAI resource → Keys and Endpoint |
| `AZURE_OPENAI_DEPLOYMENT_NAME` | Azure OpenAI Studio → Deployments |
| `AZURE_OPENAI_API_VERSION` | Azure OpenAI docs (e.g. `2024-12-01-preview`) |
| `AZURE_EMBEDDING_DEPLOYMENT` | Azure OpenAI Studio → Deployments |
| `DATABRICKS_HOST` | Databricks workspace URL |
| `DATABRICKS_TOKEN` | Databricks → User Settings → Access Tokens |



# Section 0 - Environment Setup
### Install all required packages and configure Azure credentials

**Run this section first.** If prompted to restart the kernel after installation, do so before continuing.


### Step 0.1 - Install Dependencies

In [ ]:
# ─── Install all required packages ────────────────────────────────────────
# Run this cell FIRST. Restart the kernel if prompted after installation.

%pip install -q langchain langchain-openai langchain-community \
    openai tiktoken faiss-cpu python-dotenv azure-identity \
    langchain-core langchainhub mlflow langchain-text-splitters \
    pydantic databricks-sdk databricks-langchain

### Step 0.2 - Configure Azure Credentials

Create a `.env` file in the same directory as this notebook:

```
AZURE_OPENAI_API_KEY=your_key_here
AZURE_OPENAI_ENDPOINT=https://YOUR-RESOURCE.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
AZURE_EMBEDDING_DEPLOYMENT=text-embedding-ada-002
```

Then run the cell below to load them:

In [ ]:
# ─── Load Azure credentials ────────────────────────────────────────────────
import os
from dotenv import load_dotenv

# Load from .env file if present
load_dotenv()

# ── Set your credentials here (for quick testing only — use .env in production) ──
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<TBD>.cognitiveservices.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["AZURE_EMBEDDING_DEPLOYMENT"] = "text-embedding-ada-002"

# Read into variables used throughout the lab
AZURE_API_KEY     = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_ENDPOINT    = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_DEPLOYMENT  = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
AZURE_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]

print("Credentials loaded.")
print(f"Endpoint  : {AZURE_ENDPOINT}")
print(f"Deployment: {AZURE_DEPLOYMENT}")
print(f"API Ver: {AZURE_API_VERSION}")

Credentials loaded.
Endpoint  : https://<TBD>.cognitiveservices.azure.com/
Deployment: gpt-4o
API Ver: 2024-12-01-preview


### Step 0.3 — Create the LLM Factory

This helper function creates a configured `AzureChatOpenAI` instance.  
It is **reused in every section** of this lab — define it once here, then call `get_llm()` anywhere.

| Parameter | Effect |
|-----------|--------|
| `temperature=0.0` | Deterministic output — best for extraction, compliance Q&A |
| `temperature=0.7` | More creative — better for writing, summarization |
| `max_tokens` | Hard limit on response length |

In [ ]:
# ─── Central LLM Factory — reused in every section ────────────────────────
from langchain_openai import AzureChatOpenAI

def get_llm(temperature: float = 0.0, max_tokens: int = 1024) -> AzureChatOpenAI:
    """
    Returns a configured AzureChatOpenAI instance.

    Args:
        temperature: 0.0 = deterministic, 1.0 = creative
        max_tokens:  Upper bound on response length
    """
    return AzureChatOpenAI(
        azure_deployment = AZURE_DEPLOYMENT,
        azure_endpoint   = AZURE_ENDPOINT,
        api_key          = AZURE_API_KEY,
        api_version      = AZURE_API_VERSION,
        temperature      = temperature,
        max_tokens       = max_tokens,
    )

# ── Smoke test — should print 'Azure LangChain Lab - Ready!' ──────────────
llm = get_llm()
response = llm.invoke("Reply with exactly: 'Azure LangChain Lab - Ready!'")
print(response.content)

# ── Troubleshooting ────────────────────────────────────────────────────────
# AuthenticationError  → Check AZURE_OPENAI_API_KEY
# ResourceNotFoundError → Check AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT_NAME
# RateLimitError       → Your quota is exhausted — wait or request an increase

Azure LangChain Lab - Ready!


---
# Section 1 — Azure OpenAI Endpoints
### Understand how Azure OpenAI differs from standard OpenAI and test your endpoint

Azure OpenAI endpoints use a **deployment-based URL scheme** rather than the standard `api.openai.com`.  
The key differences:

| Parameter | Standard OpenAI | Azure OpenAI |
|-----------|----------------|--------------|
| Base URL | `https://api.openai.com/v1/` | `https://YOUR-RESOURCE.openai.azure.com/` |
| Model ID | `model='gpt-4o'` | `azure_deployment='your-deployment-name'` |
| API version | Not required | `api_version='2024-12-01-preview'` |
| Auth header | `Authorization: Bearer` | `api-key` |
| LangChain class | `ChatOpenAI` | `AzureChatOpenAI` |

### Step 1.1 — Direct LLM Invocation with Message Objects

In [ ]:
# ─── Direct invocation with LangChain message objects ─────────────────────
from langchain_core.messages import HumanMessage, SystemMessage

llm = get_llm(temperature=0.3)

messages = [
    SystemMessage(content="You are an expert Azure cloud architect."),
    HumanMessage(content="What are the three main benefits of Azure Private Endpoints?")
]

response = llm.invoke(messages)

print("Response:")
print(response.content)
print()
print("Metadata:")
print(f"  Finish reason : {response.response_metadata.get('finish_reason', 'N/A')}")
print(f"  Prompt tokens : {response.usage_metadata.get('input_tokens', 'N/A')}")
print(f"  Output tokens : {response.usage_metadata.get('output_tokens', 'N/A')}")

Response:
Azure Private Endpoints provide secure and private connectivity to Azure services, ensuring that traffic remains within the Azure virtual network (VNet). The three main benefits of Azure Private Endpoints are:

### 1. **Enhanced Security**
   - Private Endpoints enable secure access to Azure services without exposing them to the public internet. They use private IP addresses from your VNet, ensuring that traffic stays entirely within your network.
   - This reduces the risk of data exfiltration and mitigates threats such as Distributed Denial of Service (DDoS) attacks and unauthorized access.
   - By eliminating public exposure, Private Endpoints help meet compliance requirements and improve your overall security posture.

### 2. **Simplified Network Architecture**
   - With Private Endpoints, you can access Azure services directly from your VNet without needing to configure complex network architectures like VPNs or ExpressRoute for secure connectivity.
   - They simplify co

### Step 1.2 — Structured JSON Output

Azure OpenAI supports **JSON-mode output**. Use `JsonOutputParser` with a Pydantic schema to extract structured data from unstructured text — useful for entity extraction, classification, and ETL pipelines.

In [ ]:
# ─── Structured JSON output with Pydantic schema ──────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define the output schema
class ExtractedEntities(BaseModel):
    companies:    List[str] = Field(description="Company names mentioned")
    technologies: List[str] = Field(description="Technology names mentioned")
    key_topics:   List[str] = Field(description="Main topics discussed")

parser = JsonOutputParser(pydantic_object=ExtractedEntities)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured entities from text. {format_instructions}"),
    ("human",  "{text}")
]).partial(format_instructions=parser.get_format_instructions())

llm = get_llm(temperature=0.0)
extraction_chain = extraction_prompt | llm | parser

sample_text = """
Azure OpenAI Service provides enterprise access to GPT-4o with SLAs and compliance.
Microsoft and Databricks have partnered to offer integrated AI solutions on Azure.
LangChain and MLflow are key components of the modern enterprise AI stack.
"""

entities = extraction_chain.invoke({"text": sample_text})
print("Extracted Entities:")
for k, v in entities.items():
    print(f"  {k}: {v}")

Extracted Entities:
  companies: ['Microsoft', 'Databricks']
  technologies: ['Azure OpenAI Service', 'GPT-4o', 'LangChain', 'MLflow']
  key_topics: ['enterprise access to AI', 'SLAs and compliance', 'integrated AI solutions', 'modern enterprise AI stack']


---
# Section 2 — LangChain LCEL Chains
### Build prompt → LLM → parser pipelines using the pipe operator

**LCEL (LangChain Expression Language)** uses the `|` pipe operator to compose chains.  
Every component — prompts, LLMs, parsers, retrievers — implements `Runnable`, making them interchangeable.

```
prompt_template  |  llm  |  output_parser
       ↓               ↓            ↓
  formats input   calls Azure   extracts text
```

**When to use:** Summarization, classification, extraction, rewriting — any single-step task.

### Step 2.1 — Basic Summarization Chain

In [ ]:
# ─── Basic LCEL chain: prompt | llm | parser ──────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.3)

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical writer. Summarize in {num_sentences} sentences."),
    ("human",  "{text}")
])

# The chain is just three components joined with |  (pipe)
summarize_chain = summarize_prompt | llm | StrOutputParser()

article = """
Large language models have transformed software development by enabling natural-language
interfaces with minimal effort. Azure OpenAI Service provides enterprise-grade access to
models like GPT-4o, with SLAs, compliance certifications (SOC 2, ISO 27001, HIPAA),
private networking via VNet, and regional data residency. LangChain abstracts the
orchestration layer, letting engineers compose multi-step AI pipelines in Python.
"""

result = summarize_chain.invoke({"text": article, "num_sentences": 2})
print("Summary:")
print(result)

Summary:
Large language models, like those in Azure OpenAI Service, have revolutionized software development by offering enterprise-grade features such as compliance certifications, private networking, and regional data residency. LangChain simplifies AI pipeline creation by providing an orchestration layer for multi-step workflows in Python.


### Step 2.2 — Using .batch() for Multiple Inputs

`.batch()` runs the same chain on multiple inputs **concurrently**, which is much faster than a Python for-loop.

In [ ]:
# ─── Batch processing — runs all inputs concurrently ──────────────────────
texts = [
    {"text": "Azure Kubernetes Service (AKS) simplifies container orchestration on Azure.", "num_sentences": 1},
    {"text": "Azure SQL Managed Instance is a fully managed PaaS database service.",          "num_sentences": 1},
    {"text": "Azure Blob Storage provides massively scalable object storage for any data.",   "num_sentences": 1},
]

results = summarize_chain.batch(texts)

print("Batch Results:")
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r}")

Batch Results:
  [1] Azure Kubernetes Service (AKS) provides a managed Kubernetes platform to streamline container deployment, scaling, and management on Azure.
  [2] Azure SQL Managed Instance is a fully managed Platform-as-a-Service (PaaS) database solution that combines the features of SQL Server with the benefits of a cloud-based environment.
  [3] Azure Blob Storage offers highly scalable object storage for diverse data types.


---
# Section 3 — Sequential Chains
### Pass the output of one chain as the input to the next

A **sequential chain** models multi-step workflows:
```
[Document] → [Summarize] → [Critique] → [Executive Brief] → Final Output
```

Use `RunnablePassthrough.assign()` to **accumulate intermediate values** in a shared state dictionary.

**When to use:** Document processing pipelines, multi-stage reasoning, content transformation.

### Step 3.1 — Three-Step Document Pipeline

In [ ]:
# ─── Sequential chain with intermediate state ──────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = get_llm(temperature=0.4)
str_parser = StrOutputParser()

# ── Step 1: Summarize ──────────────────────────────────────────────────────
step1_chain = (
    ChatPromptTemplate.from_template(
        "Summarize this technical document in 3 bullet points:\n\n{document}"
    ) | llm | str_parser
)

# ── Step 2: Critique ───────────────────────────────────────────────────────
step2_chain = (
    ChatPromptTemplate.from_template(
        "Given this summary:\n{summary}\n\n"
        "List 2 potential risks or gaps a senior architect should investigate."
    ) | llm | str_parser
)

# ── Step 3: Executive brief ────────────────────────────────────────────────
step3_chain = (
    ChatPromptTemplate.from_template(
        "Summary:\n{summary}\n\nRisks:\n{risks}\n\n"
        "Write a 2-sentence executive brief for a VP-level audience."
    ) | llm | str_parser
)

# ── Compose the full pipeline ──────────────────────────────────────────────
# Each .assign() adds a new key to the state dictionary
full_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(summary = step1_chain)
    | RunnablePassthrough.assign(risks   = step2_chain)
    | step3_chain
)

rfc_text = """
RFC-2024-091: Migration of on-premises Oracle DB to Azure SQL Managed Instance.
Phase 1 covers schema migration using Database Migration Service (DMS).
Phase 2 covers data migration with minimal downtime using transactional replication.
Phase 3 covers cutover and decommission. Estimated downtime window: 4 hours.
Dependencies: VNet peering, ExpressRoute, legacy ETL jobs on SSIS.
"""

brief = full_pipeline.invoke(rfc_text)
print("Executive Brief:")
print(brief)

Executive Brief:
The migration of the on-premises Oracle database to Azure SQL Managed Instance is planned in three phases, leveraging Azure Database Migration Service and transactional replication to ensure minimal downtime, with a 4-hour cutover window. Key risks include potential network connectivity issues with VNet peering and ExpressRoute, as well as the need to validate and optimize legacy SSIS ETL jobs for compatibility and performance in the new environment.


### Step 3.2 — Inspect Intermediate Steps (Debug Mode)

In [ ]:
# ─── Debug pipeline — keep all intermediate values ─────────────────────────
debug_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(summary = step1_chain)
    | RunnablePassthrough.assign(risks   = step2_chain)
    | RunnablePassthrough.assign(brief   = step3_chain)
)

full_state = debug_pipeline.invoke(rfc_text)

print("=" * 60)
print("STEP 1 — Summary:")
print(full_state["summary"])
print("\nSTEP 2 — Risks:")
print(full_state["risks"])
print("\nSTEP 3 — Brief:")
print(full_state["brief"])

STEP 1 — Summary:
- **Migration Phases**: The document outlines three phases for migrating an on-premises Oracle database to Azure SQL Managed Instance—schema migration (Phase 1), data migration with minimal downtime via transactional replication (Phase 2), and cutover with decommissioning (Phase 3). The estimated downtime during cutover is 4 hours.  

- **Key Dependencies**: The migration process depends on VNet peering, ExpressRoute connectivity, and addressing legacy ETL jobs on SSIS.  

- **Tools and Approach**: Microsoft Database Migration Service (DMS) is used for schema migration, and transactional replication ensures minimal downtime during data migration.

STEP 2 — Risks:
1. **Network Connectivity and Latency Issues**: The migration process heavily depends on VNet peering and ExpressRoute connectivity. A senior architect should investigate potential risks related to network latency, bandwidth limitations, or misconfigurations that could impact transactional replication perform

Please re-run the above step if facing any 400/401 error.

---
# Section 4 — Router Chains
### Classify the input and route it to the best specialized sub-chain

A **router chain** uses an LLM classifier to pick the most appropriate chain for each query:

```
Input ──▶ [Classifier] ──▶ "legal"   ──▶ [Legal Expert Chain]
                           "finance" ──▶ [Finance Expert Chain]
                           "hr"      ──▶ [HR Expert Chain]
                           "other"   ──▶ [General Chain]
```

**When to use:** Enterprise chatbots, multi-domain Q&A, support ticket triage.

### Step 4.1 — Build the Router

In [ ]:
# ─── Router chain with domain-specific sub-chains ──────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough, RunnableLambda

llm = get_llm(temperature=0.2)
str_out = StrOutputParser()

# ── Specialized sub-chains ─────────────────────────────────────────────────
legal_chain = (
    ChatPromptTemplate.from_template(
        "You are a corporate legal advisor. Answer this legal question concisely:\n{query}"
    ) | llm | str_out
)

finance_chain = (
    ChatPromptTemplate.from_template(
        "You are a finance director. Explain the accounting treatment and business impact:\n{query}"
    ) | llm | str_out
)

hr_chain = (
    ChatPromptTemplate.from_template(
        "You are an HR Business Partner. Answer with empathy and policy awareness:\n{query}"
    ) | llm | str_out
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Answer the following question helpfully and accurately:\n{query}"
    ) | llm | str_out
)

# ── Classifier chain ───────────────────────────────────────────────────────
classifier_prompt = ChatPromptTemplate.from_template(
    """Classify the following query into exactly ONE category: legal, finance, hr, other.
Respond with just the single word, lowercase.

Query: {query}
Category:"""
)
classifier = classifier_prompt | llm | str_out

# ── Router using RunnableBranch ────────────────────────────────────────────
router = RunnableBranch(
    (lambda x: "legal"   in x["category"], legal_chain),
    (lambda x: "finance" in x["category"], finance_chain),
    (lambda x: "hr"      in x["category"], hr_chain),
    general_chain,   # default fallback
)

# ── Full routed pipeline ───────────────────────────────────────────────────
routed_pipeline = (
    RunnablePassthrough.assign(category = classifier)
    | RunnablePassthrough.assign(
          route_used = lambda x: print(f"→ Routed to: [{x['category'].strip()}]") or x["category"]
      )
    | router
)

print("Router pipeline ready.")

Router pipeline ready.


### Step 4.2 — Test the Router on Different Query Types

In [ ]:
# ─── Test router with queries from different domains ──────────────────────
queries = [
    "What are the GDPR implications of storing customer PII in a US-based Azure region?",
    "How should we account for cloud infrastructure CapEx vs OpEx in our P&L?",
    "Our employee is requesting a hybrid work accommodation. What is our obligation?",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Query: {q[:80]}")
    answer = routed_pipeline.invoke({"query": q})
    print(f"Answer: {answer[:300]}...")


Query: What are the GDPR implications of storing customer PII in a US-based Azure regio
→ Routed to: [legal]
Answer: Storing customer Personally Identifiable Information (PII) in a US-based Azure region has significant GDPR implications, as the GDPR requires that data transfers outside the EU/EEA comply with its rules on international data transfers. Specifically:

1. **Adequacy Decision**: The US does not current...

Query: How should we account for cloud infrastructure CapEx vs OpEx in our P&L?
→ Routed to: [finance]
Answer: As a finance director, understanding the accounting treatment and business impact of cloud infrastructure expenditures is critical for accurate financial reporting and strategic decision-making. Here's a breakdown of how to account for cloud infrastructure CapEx (capital expenditure) versus OpEx (op...

Query: Our employee is requesting a hybrid work accommodation. What is our obligation?
→ Routed to: [hr]
Answer: Thank you for bringing this to my attention. Sup

---
# Section 5 — Memory & Stateful Conversations
### Add chat history to chains for multi-turn conversations

LLMs are **stateless by default** — each call starts fresh. LangChain v0.3+ uses  
`RunnableWithMessageHistory` + a message history store to add state.

| Old API (removed) | Modern Replacement | Best For |
|---|---|---|
| `ConversationBufferMemory` | `InMemoryChatMessageHistory` | Short sessions |
| `ConversationSummaryMemory` | `trim_messages(strategy="last")` | Long sessions |
| `ConversationBufferWindowMemory` | `trim_messages(max_messages=N)` | Keep last N turns |

### Step 5.1 — Build a Stateful Conversation Chain

In [ ]:
# ─── Stateful multi-turn conversation with message history ─────────────────
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages, BaseMessage
from langchain_core.output_parsers import StrOutputParser
from typing import List
import tiktoken

llm = get_llm(temperature=0.5)

# ── Token counter for gpt-4o (uses o200k_base encoding) ───────────────────
def count_tokens(messages: List[BaseMessage]) -> int:
    enc   = tiktoken.get_encoding("o200k_base")  # gpt-4o encoding
    total = 0
    for msg in messages:
        total += 4 + len(enc.encode(str(msg.content)))  # 4 = per-message overhead
    return total

# Keep last 500 tokens of history
trimmer = trim_messages(
    max_tokens    = 500,
    strategy      = "last",
    token_counter = count_tokens,
    include_system= True,
    allow_partial = False,
)

# ── Prompt with chat history placeholder ──────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a seasoned Azure solutions architect helping a team plan their cloud migration. "
     "Be specific and professional. Reference prior conversation context when relevant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# ── In-memory session store (swap for Redis/CosmosDB in production) ────────
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain = prompt | trimmer | llm | StrOutputParser()

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key   = "input",
    history_messages_key = "chat_history",
)

print("Conversation chain with memory ready.")

Conversation chain with memory ready.


### Step 5.2 — Run a Multi-Turn Conversation

In [ ]:
# ─── Multi-turn conversation — each turn references prior context ──────────
SESSION = {"configurable": {"session_id": "azure-migration-01"}}

turns = [
    "We're migrating 40 microservices from AWS EKS to Azure Kubernetes Service. Where do we start?",
    "Our services use Kafka heavily. What's the Azure equivalent and migration path?",
    "Given our Kafka dependency you mentioned, what networking topology do you recommend?",
    "Summarize our architecture decisions so far in a table.",
]

for i, turn in enumerate(turns, 1):
    print(f"\n{'─'*60}")
    print(f"Turn {i}: {turn}")
    reply = chain_with_memory.invoke({"input": turn}, config=SESSION)
    print(f"Assistant: {reply[:400]}...")


────────────────────────────────────────────────────────────
Turn 1: We're migrating 40 microservices from AWS EKS to Azure Kubernetes Service. Where do we start?
Assistant: Migrating microservices from AWS EKS (Elastic Kubernetes Service) to Azure Kubernetes Service (AKS) is a significant undertaking that requires careful planning and execution. Here’s a structured approach to get started:

---

### **1. Assess and Plan**
#### **a. Inventory and Dependency Mapping**
- Create a detailed inventory of the microservices running on EKS, including:
  - Service configuratio...

────────────────────────────────────────────────────────────
Turn 2: Our services use Kafka heavily. What's the Azure equivalent and migration path?
Assistant: Azure provides **Azure Event Hubs** as its equivalent to Apache Kafka for streaming data ingestion and processing. Event Hubs is a highly scalable event streaming platform and supports Kafka protocol natively, which makes migrating Kafka-based workloads to Azu

---
# Section 6 — Build a Vector Store
### Index your documents with Azure embeddings for semantic search

A **vector store** converts text into numeric vectors (embeddings) and enables semantic search —  
finding documents by *meaning* rather than exact keyword match.

```
Documents → [Chunker] → Chunks → [Embeddings Model] → Vectors → [FAISS Index]
                                                                       ↑
Query → [Embeddings Model] → Query Vector ──────────────────── Similarity Search
```

| Concept | Explanation |
|---------|-------------|
| Embedding | Numeric vector (1536 dims) representing semantic meaning |
| Chunking | Split docs so each embedding captures a focused idea |
| FAISS | Local vector store — fast, in-process |
| Azure AI Search | Managed vector search — recommended for production |

### Step 6.1 — Configure Azure Embeddings

In [ ]:
# ─── Configure Azure OpenAI Embeddings ────────────────────────────────────
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_deployment = os.environ.get("AZURE_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"),
    azure_endpoint   = AZURE_ENDPOINT,
    api_key          = AZURE_API_KEY,
    api_version      = AZURE_API_VERSION,
)

# Quick test — embed a sentence and check the vector dimensions
test_vec = embeddings.embed_query("Hello Azure")
print(f"Embedding model ready. Vector dimensions: {len(test_vec)}")
# Expected: 1536 for text-embedding-ada-002

Embedding model ready. Vector dimensions: 1536


### Step 6.2 — Load and Chunk Documents

In [ ]:
# ─── Load internal knowledge base and split into chunks ───────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Sample internal policy documents (replace with your real documents in production)
internal_docs = [
    Document(
        page_content="""
        Azure Policy: Cloud Spend Governance v2.3 (2024)
        All Azure resources must be tagged with: CostCenter, Environment, Owner.
        Monthly cloud spend over $50,000 requires VP Finance approval.
        Reserved Instances must be evaluated quarterly by the FinOps team.
        Orphaned resources (no activity >30 days) are auto-flagged for deletion.
        """,
        metadata={"source": "cloud_policy_v2.3", "type": "policy"}
    ),
    Document(
        page_content="""
        Azure Architecture Standard: Networking (2024)
        All production workloads must use Private Endpoints — no public internet exposure.
        Hub-and-spoke topology is the approved network design for enterprise subscriptions.
        Azure Firewall Premium is mandatory for all outbound traffic from production VNets.
        ExpressRoute is required for any workload transferring >1TB/day to on-premises.
        """,
        metadata={"source": "arch_network_2024", "type": "standard"}
    ),
    Document(
        page_content="""
        Data Classification Policy (Effective Jan 2024)
        Level 1 (Public): No restrictions.
        Level 2 (Internal): Encryption at rest required.
        Level 3 (Confidential): Encryption at rest + in transit + access logging.
        Level 4 (Restricted): All Level 3 + Customer Managed Keys (CMK)
                               + Defender for Cloud + quarterly penetration testing.
        PII and financial data are always classified as Level 4.
        """,
        metadata={"source": "data_classification_2024", "type": "policy"}
    ),
]

# Split into 300-character chunks with 50-character overlap
# Smaller chunks = more focused embeddings = better retrieval precision
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks   = splitter.split_documents(internal_docs)

print(f"Created {len(chunks)} chunks from {len(internal_docs)} documents")

Created 6 chunks from 3 documents


### Step 6.3 — Build and Query the FAISS Index

In [ ]:
# ─── Build FAISS vector store and test semantic search ────────────────────
from langchain_community.vectorstores import FAISS

# Build the index — calls Azure embeddings API for each chunk
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks")

# ── Test semantic search ───────────────────────────────────────────────────
query   = "Do we need encryption for salary data?"
results = vectorstore.similarity_search(query, k=2)

print(f"\nQuery: {query}")
for i, doc in enumerate(results, 1):
    print(f"\nResult {i} — Source: {doc.metadata['source']}")
    print(doc.page_content.strip()[:200])

# ── Persist the index to disk (no re-embedding on restart) ────────────────
vectorstore.save_local("faiss_index")
print("\nIndex saved to ./faiss_index/")

# To load it back:
# vectorstore = FAISS.load_local("faiss_index", embeddings,
#                                allow_dangerous_deserialization=True)

Indexed 6 chunks

Query: Do we need encryption for salary data?

Result 1 — Source: data_classification_2024
Data Classification Policy (Effective Jan 2024)
        Level 1 (Public): No restrictions.
        Level 2 (Internal): Encryption at rest required.
        Level 3 (Confidential): Encryption at rest +

Result 2 — Source: data_classification_2024
Level 4 (Restricted): All Level 3 + Customer Managed Keys (CMK)
                               + Defender for Cloud + quarterly penetration testing.
        PII and financial data are always classifie

Index saved to ./faiss_index/


---
# Section 7 — RAG Chain (Retrieval-Augmented Generation)
### Ground LLM responses in your private documents

**RAG** eliminates hallucinations about internal knowledge by combining a retriever  
(your vector store) with an LLM:

```
Query ──▶ [Embeddings] ──▶ [Vector Search] ──▶ Relevant Chunks
                                                      ↓
                                         [LLM + Context] ──▶ Answer
```

**When to use:** Internal knowledge bases, compliance Q&A, document-grounded chatbots.

> 💡 **This section requires Sections 6 to have been run first** (vectorstore and retriever must be defined).

### Step 7.1 — Build the RAG Chain

In [ ]:
# ─── RAG chain: retriever + Azure OpenAI LLM ──────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.0)  # Deterministic — best for compliance answers

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an enterprise Azure compliance advisor.
Answer questions ONLY based on the provided internal policy documents.
If the answer is not in the context, say so explicitly — do not invent.
Always cite the source document name.

Context from internal policies:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', 'unknown')}]\n{d.page_content.strip()}"
        for d in docs
    )

# The RAG chain:
#  1. Run the question through the retriever to get relevant chunks
#  2. Format the chunks as context
#  3. Pass context + question to the LLM
#  4. Parse the response to a string
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


### Step 7.2 — Test the RAG Chain

In [ ]:
# ─── Test RAG with compliance questions ───────────────────────────────────
questions = [
    "Do we need Customer Managed Keys for a database storing employee salary data?",
    "What approvals do I need to spin up $80K/month of Azure resources?",
    "Can we expose our production API directly to the internet?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print()


Q: Do we need Customer Managed Keys for a database storing employee salary data?
A: Yes, you need Customer Managed Keys (CMK) for a database storing employee salary data because financial data is always classified as Level 4, which requires CMK as per the **data_classification_2024** policy.

Source: **data_classification_2024**


Q: What approvals do I need to spin up $80K/month of Azure resources?
A: According to the "Cloud Spend Governance v2.3 (2024)" policy, monthly cloud spend over $50,000 requires VP Finance approval. Therefore, you will need approval from the VP Finance to spin up $80K/month of Azure resources.

Source: [cloud_policy_v2.3]


Q: Can we expose our production API directly to the internet?
A: No, production workloads must use Private Endpoints and cannot be exposed directly to the public internet. This is stated in the "Azure Architecture Standard: Networking (2024)" document. 

Source: [arch_network_2024]



---
# Section 8 — MLflow Tracking
### Log experiments, compare runs, and register your RAG model

**MLflow** is the industry standard for ML experiment tracking. Use it to:  
- Compare different chunk sizes, retrieval strategies, and prompt templates  
- Log latency, token usage, and answer quality metrics  
- Register the best chain for deployment via Model Serving

### Step 8.1

In [ ]:
# ─── Step 1: Write the chain definition to a file ─────────────────────────
import os

chain_code = '''
import os
import mlflow
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

embeddings = AzureOpenAIEmbeddings(
    azure_deployment = os.environ.get("AZURE_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"),
    azure_endpoint   = os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key          = os.environ["AZURE_OPENAI_API_KEY"],
    api_version      = os.environ["AZURE_OPENAI_API_VERSION"],
)

internal_docs = [
    Document(page_content="""
        Azure Policy: Cloud Spend Governance v2.3
        All Azure resources must be tagged: CostCenter, Environment, Owner.
        Monthly cloud spend over $50,000 requires VP Finance approval.
        Orphaned resources (no activity >30 days) are auto-flagged for deletion.
    """, metadata={"source": "cloud_policy_v2.3"}),
    Document(page_content="""
        Azure Architecture Standard: Networking
        All production workloads must use Private Endpoints.
        Hub-and-spoke topology is required for enterprise subscriptions.
    """, metadata={"source": "arch_network_2024"}),
    Document(page_content="""
        Data Classification Policy
        Level 4 (Restricted): Encryption at rest + in transit + Customer Managed Keys.
        PII and financial data are always classified as Level 4.
    """, metadata={"source": "data_classification_2024"}),
]

chunks    = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(internal_docs)
vs        = FAISS.from_documents(chunks, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})

llm = AzureChatOpenAI(
    azure_deployment = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    azure_endpoint   = os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key          = os.environ["AZURE_OPENAI_API_KEY"],
    api_version      = os.environ["AZURE_OPENAI_API_VERSION"],
    temperature      = 0.0,
    max_tokens       = 1024,
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an enterprise Azure compliance advisor.
Answer ONLY from the provided internal policy documents.
If the answer is not in the context, say so — do not invent.
Always cite the source document name.

Context:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\\n\\n".join(
        f"[{d.metadata.get(\'source\', \'unknown\')}]\\n{d.page_content.strip()}"
        for d in docs
    )

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Required — tells MLflow which object to serve
mlflow.models.set_model(chain)
'''

os.makedirs("rag_chain_code", exist_ok=True)
with open("rag_chain_code/chain.py", "w") as f:
    f.write(chain_code)

print("Chain file written to rag_chain_code/chain.py")

Chain file written to rag_chain_code/chain.py


### Step 8.2

In [ ]:
# ─── Step 2: Log and register using the file path (not the object) ────────
import mlflow.langchain

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("rag_experiments")

with mlflow.start_run(run_name="rag_v1_registered"):
    model_info = mlflow.langchain.log_model(
        lc_model              = "rag_chain_code/chain.py",  # ← path, NOT the rag_chain object
        name                  = "rag_chain",
        registered_model_name = "azure_rag_chain",
        input_example         = {"question": "Do we need CMK for salary data?"},
    )
    print(f"Model registered: {model_info.model_uri}")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/11 17:36:43 WARNING mlflow.models.signature: Failed to infer model output schema from prediction result, setting output schema to AnyType. For full traceback, set logging level to debug.
2026/03/11 17:37:13 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `con

Model registered: models:/m-b3cf7e7ceeba41e9a5f5332e4140d6f2


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Registered model 'azure_rag_chain' already exists. Creating a new version of this model...
Created version '3' of model 'azure_rag_chain'.


### Step 8.3 — Register the RAG Chain as an MLflow Model

In [ ]:
# ─── Step 3: Load and test the registered model ───────────────────────────
loaded_chain = mlflow.langchain.load_model(model_info.model_uri)
result = loaded_chain.invoke("What tagging is required for Azure resources?")
print(f"\nLoaded model answer: {result}")


Loaded model answer: All Azure resources must be tagged with: CostCenter, Environment, and Owner. 

Source: [cloud_policy_v2.3]


### Step 8.4 — Compare Experiment Runs

In [ ]:
# ─── Compare runs across the experiment ───────────────────────────────────
import mlflow
import pandas as pd

mlflow.set_tracking_uri("mlruns")

runs = mlflow.search_runs(
    experiment_names = ["rag_experiments"],
    order_by         = ["metrics.avg_latency_sec ASC"]
)

if runs.empty:
    print("No runs found — complete Steps 8.2 and 8.3 first.")
else:
    # ── Pick only columns that actually exist in this run set ─────────────
    wanted_cols = {
        "tags.mlflow.runName"    : "Run Name",
        "params.llm_model"       : "LLM Model",
        "params.chunk_size"      : "Chunk Size",
        "params.retriever_k"     : "Retriever K",
        "params.temperature"     : "Temperature",
        "metrics.avg_latency_sec": "Avg Latency (s)",
        "metrics.max_latency_sec": "Max Latency (s)",
        "metrics.num_questions"  : "Questions",
        "status"                 : "Status",
    }

    available = {k: v for k, v in wanted_cols.items() if k in runs.columns}
    display_df = runs[list(available.keys())].rename(columns=available)

    # ── Round numeric columns for readability ─────────────────────────────
    for col in ["Avg Latency (s)", "Max Latency (s)"]:
        if col in display_df.columns:
            display_df[col] = pd.to_numeric(display_df[col], errors="coerce").round(2)

    print(f"Found {len(runs)} run(s) in experiment 'rag_experiments'\n")
    print(display_df.to_string(index=False))

    # ── Best run summary ──────────────────────────────────────────────────
    if "Avg Latency (s)" in display_df.columns:
        best = display_df.loc[display_df["Avg Latency (s)"].idxmin()]
        print(f"\nFastest run : {best.get('Run Name', 'N/A')}  "
              f"({best['Avg Latency (s)']}s avg latency)")

print("\nView full UI:")
print("   Local      → run 'mlflow ui' in terminal, open http://localhost:5000")
print("   Databricks → click 'Experiments' in the sidebar")

Found 8 run(s) in experiment 'rag_experiments'

         Run Name LLM Model Chunk Size Retriever K Temperature  Avg Latency (s)  Max Latency (s)  Questions   Status
  rag_v1_chunk300    gpt-4o        300           3         0.0             1.62             1.92        2.0 FINISHED
rag_v1_registered      None       None        None        None              NaN              NaN        NaN FINISHED
rag_v1_registered      None       None        None        None              NaN              NaN        NaN FINISHED
rag_v1_registered      None       None        None        None              NaN              NaN        NaN FINISHED
rag_v1_registered      None       None        None        None              NaN              NaN        NaN   FAILED
rag_v1_registered      None       None        None        None              NaN              NaN        NaN   FAILED
rag_v1_registered      None       None        None        None              NaN              NaN        NaN   FAILED
rag_v1_registere

---
# Section 9 — Databricks RAG Pipeline
### Production-scale RAG with Unity Catalog and Mosaic AI Vector Search

Databricks provides a managed environment for RAG at scale:
- **Unity Catalog** — document governance and versioning  
- **Mosaic AI Vector Search** — managed, auto-syncing vector index  
- **MLflow** — experiment tracking and model registry (integrated)

> ⚠️ **Prerequisites for this section:**
> - Databricks workspace with Mosaic AI enabled (Azure Databricks Premium)
> - Unity Catalog enabled
> - `DATABRICKS_HOST` and `DATABRICKS_TOKEN` environment variables set

In [ ]:
# ─── Check available Databricks model endpoints ───────────────────────────
from databricks.sdk import WorkspaceClient
import os

DATABRICKS_HOST  = os.environ.get("DATABRICKS_HOST",  "https://adb-7405608086884528.8.azuredatabricks.net/")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN", "your_databricks_token_here")

w = WorkspaceClient(
    host  = DATABRICKS_HOST,
    token = DATABRICKS_TOKEN,
)

print("── Available Model Serving Endpoints ─────────────────────────────────")
for endpoint in w.serving_endpoints.list():
    print(f"  {endpoint.name} — state: {endpoint.state.ready.value}")

── Available Model Serving Endpoints ─────────────────────────────────
  databricks-gpt-oss-120b — state: READY
  databricks-gpt-oss-20b — state: READY
  databricks-qwen3-next-80b-a3b-instruct — state: READY
  databricks-llama-4-maverick — state: READY
  databricks-gemma-3-12b — state: READY
  databricks-gte-large-en — state: READY
  databricks-bge-large-en — state: READY
  databricks-meta-llama-3-1-8b-instruct — state: READY
  databricks-meta-llama-3-3-70b-instruct — state: READY
  databricks-qwen3-embedding-0-6b — state: READY
  databricks-meta-llama-3-1-405b-instruct — state: READY


In [ ]:
import os
import mlflow
from dotenv import load_dotenv
load_dotenv()

# ── Must be set BEFORE importing ChatDatabricks ────────────────────────────
os.environ["DATABRICKS_HOST"]  = DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
mlflow.set_tracking_uri("databricks")

from databricks_langchain import ChatDatabricks
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Step 1: Databricks LLM ────────────────────────────────────────────────
llm = ChatDatabricks(
    endpoint    = "databricks-meta-llama-3-3-70b-instruct",
    temperature = 0.0,
    max_tokens  = 1024,
)

test = llm.invoke("Reply with exactly: Databricks LLM Ready")
print(f"Databricks LLM  : {test.content}")

# ── Step 2: Azure OpenAI embeddings ───────────────────────────────────────
embeddings = AzureOpenAIEmbeddings(
    azure_deployment = "text-embedding-ada-002",
    azure_endpoint   = "https://<TBD>.cognitiveservices.azure.com/",
    api_key          = "",
    api_version      = "2024-12-01-preview",
)

test_vec = embeddings.embed_query("Hello Azure")
print(f"Azure Embeddings : {len(test_vec)} dimensions")

# ── Step 3: Documents ─────────────────────────────────────────────────────
docs = [
    Document(page_content="""
        Azure Policy: Cloud Spend Governance v2.3
        All Azure resources must be tagged: CostCenter, Environment, Owner.
        Monthly cloud spend over $50,000 requires VP Finance approval.
        Orphaned resources with no activity over 30 days are auto-flagged for deletion.
    """, metadata={"source": "cloud_policy_v2.3"}),
    Document(page_content="""
        Azure Architecture Standard: Networking
        All production workloads must use Private Endpoints — no public internet exposure.
        Hub-and-spoke topology is required for all enterprise subscriptions.
        Azure Firewall Premium is mandatory for outbound production traffic.
    """, metadata={"source": "arch_network_2024"}),
    Document(page_content="""
        Data Classification Policy (Jan 2024)
        Level 1 Public      : No restrictions.
        Level 2 Internal    : Encryption at rest required.
        Level 3 Confidential: Encryption at rest and in transit.
        Level 4 Restricted  : Encryption + Customer Managed Keys (CMK)
                              + Defender for Cloud + quarterly pen testing.
        PII and financial data are always classified as Level 4.
    """, metadata={"source": "data_classification_2024"}),
    Document(page_content="""
        Incident Response Policy
        All P1 incidents must be declared within 15 minutes of detection.
        A war room must be opened in Microsoft Teams within 30 minutes.
        RCA must be submitted within 5 business days of resolution.
        Security incidents must also notify the CISO within 1 hour.
    """, metadata={"source": "incident_response_2024"}),
]

# ── Step 4: FAISS index ───────────────────────────────────────────────────
chunks      = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(docs)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"FAISS index      : {len(chunks)} chunks indexed")

# ── Step 5: RAG chain ─────────────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an enterprise Azure compliance advisor.
Answer ONLY from the provided context — do not invent answers.
Always cite the source document at the end of your answer.

Context:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', 'unknown')}]\n{d.page_content.strip()}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready\n")

# ── Step 6: Test ──────────────────────────────────────────────────────────
questions = [
    "Do we need Customer Managed Keys for a salary database?",
    "What approvals do I need to spend $80K/month on Azure?",
    "Can we expose our production API directly to the internet?",
    "How quickly must we declare a P1 incident?",
]

for q in questions:
    print(f"{'─'*60}")
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print()

✅ Databricks LLM  : Databricks LLM Ready
✅ Azure Embeddings : 1536 dimensions
✅ FAISS index      : 6 chunks indexed
✅ RAG chain ready

────────────────────────────────────────────────────────────
Q: Do we need Customer Managed Keys for a salary database?
A: For a salary database, which typically contains sensitive information, it would be classified as Level 4 Restricted according to the Data Classification Policy. This is because salary information is highly sensitive and requires the highest level of protection. 

According to the policy, Level 4 Restricted data requires encryption and Customer Managed Keys (CMK). 

Source: [data_classification_2024]

────────────────────────────────────────────────────────────
Q: What approvals do I need to spend $80K/month on Azure?
A: To spend $80K/month on Azure, you need VP Finance approval since the monthly cloud spend exceeds $50,000. 

Source: [cloud_policy_v2.3] Azure Policy: Cloud Spend Governance v2.3

─────────────────────────────────────

---
# Section 10 — Agents & Tools
### Let the LLM decide which tool to call and in what order

An **agent** runs a ReAct (Reasoning + Acting) loop:
```
User Goal ──▶ [LLM Thinks] ──▶ Tool Call ──▶ [LLM Observes] ──▶ Next Tool / Final Answer
                 ↑_______________________________↓
                           (ReAct loop)
```

**When to use:** Complex Q&A requiring multiple lookups, automated FinOps, code execution.

### Step 10.1 — Define Custom Enterprise Tools

In [ ]:
# ─── Define enterprise tools for the agent ────────────────────────────────
from langchain_core.tools import tool
import json, datetime, random

def parse_input(raw):
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            pass
        result = {}
        for part in raw.split(","):
            part = part.strip()
            if "=" in part:
                k, v = part.split("=", 1)
                v = v.strip().strip("'\"")
                try:    v = int(v)
                except ValueError:
                    try: v = float(v)
                    except ValueError: pass
                result[k.strip()] = v
        if result:
            return result
    return raw

@tool
def get_azure_resource_cost(resource_group: str, days: int = 30) -> str:
    """Get Azure resource cost for a resource group over the past N days.
    Use when asked about cloud spending or costs for a specific resource group."""
    cost  = round(random.uniform(1000, 50000), 2)  # Replace with real Azure Cost Management API
    trend = random.choice(["up 12%", "down 8%", "stable"])
    return json.dumps({
        "resource_group"        : resource_group,
        "period_days"           : days,
        "total_cost_usd"        : cost,
        "trend_vs_prior_period" : trend,
        "top_services"          : ["Azure Kubernetes Service", "Azure SQL MI", "Blob Storage"]
    })

@tool
def check_compliance_status(subscription_id: str) -> str:
    """Check Azure Policy compliance status for a subscription.
    Use when asked about policy violations or compliance posture."""
    violations = random.randint(0, 15)
    return json.dumps({
        "subscription_id"     : subscription_id,
        "compliance_score"    : f"{random.randint(70, 99)}%",
        "total_policies"      : 124,
        "violations"          : violations,
        "critical_violations" : random.randint(0, min(3, violations)),
        "last_evaluated"      : datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    })

@tool
def list_orphaned_resources(resource_group: str) -> str:
    """List Azure resources with no activity in the past 30 days.
    Use to identify waste or cleanup opportunities."""
    resources = [
        {"name": "vm-dev-unused-01",   "type": "Virtual Machine", "idle_days": 45, "monthly_cost": 312},
        {"name": "disk-backup-old-02", "type": "Managed Disk",    "idle_days": 91, "monthly_cost": 48},
        {"name": "pip-staging-03",     "type": "Public IP",       "idle_days": 33, "monthly_cost": 3.5},
    ]
    return json.dumps({
        "resource_group"           : resource_group,
        "orphaned_resources"       : resources,
        "potential_monthly_savings": sum(r["monthly_cost"] for r in resources)
    })

@tool
def calculate_reserved_instance_savings(input: str) -> str:
    """Calculate Azure Reserved Instance savings vs pay-as-you-go.
    Input: JSON with vm_family, quantity, term_years e.g. {"vm_family": "D4s_v3", "quantity": 20, "term_years": 1}"""
    parsed     = parse_input(input)
    vm_family  = parsed.get("vm_family",  "D4s_v3") if isinstance(parsed, dict) else "D4s_v3"
    quantity   = int(parsed.get("quantity",   20)   if isinstance(parsed, dict) else 20)
    term_years = int(parsed.get("term_years",  1)   if isinstance(parsed, dict) else 1)
    payg       = quantity * random.uniform(200, 800)
    discount   = 0.35 if term_years == 1 else 0.55
    return json.dumps({
        "vm_family"         : vm_family,
        "quantity"          : quantity,
        "term_years"        : term_years,
        "payg_monthly_usd"  : round(payg, 2),
        "ri_monthly_usd"    : round(payg * (1 - discount), 2),
        "discount_pct"      : f"{discount*100:.0f}%",
        "annual_savings_usd": round(payg * discount * 12, 2)
    })

tools = [get_azure_resource_cost, check_compliance_status,
         list_orphaned_resources, calculate_reserved_instance_savings]

print("Tools registered:")
for t in tools:
    print(f"   {t.name}")

Tools registered:
   get_azure_resource_cost
   check_compliance_status
   list_orphaned_resources
   calculate_reserved_instance_savings


### Step 10.2 — Create the Agent

In [ ]:
# ── ReAct prompt — must have these 4 variables ────────────────────────────
# {tools}, {tool_names}, {input}, {agent_scratchpad} are all required

from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""You are an Azure FinOps and compliance expert.
Always call tools to gather data before answering. Never answer without calling tools first.

You have access to the following tools:
{tools}

Use this format strictly:
Question: the input question you must answer
Thought: think about what to do
Action: the tool to use, must be one of [{tool_names}]
Action Input: the input to the tool
Observation: the result of the tool
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now have enough data to give a final answer
Final Answer: the complete report with all retrieved data

Begin!

Question: {input}
Thought: {agent_scratchpad}""")

# ── Create agent ──────────────────────────────────────────────────────────
agent          = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(
    agent                = agent,
    tools                = tools,
    verbose              = True,
    handle_parsing_errors= True,
    max_iterations       = 8,
)

print(f"Agent ready with {len(tools)} tools\n")

Agent ready with 4 tools



### Step 10.3 — Run the Agent

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────
result = agent_executor.invoke({
    "input": """
    Provide a cloud optimization report for resource group 'prod-eastus-rg'
    in subscription 'sub-prod-001'. Include:
    1. Current spend and trend
    2. Orphaned resources and potential savings
    3. Compliance posture
    4. RI savings for 20x D4s_v3 VMs on a 1-year term
    5. Prioritized action plan with total estimated savings
    """
})

print("\n" + "="*60)
print("FINAL REPORT:")
print("="*60)
print(result["output"])



> Entering new AgentExecutor chain...
To provide a comprehensive cloud optimization report for the 'prod-eastus-rg' resource group in the 'sub-prod-001' subscription, I need to gather data on current spend, orphaned resources, compliance posture, and potential Reserved Instance (RI) savings.

Action: get_azure_resource_cost
Action Input: resource_group='prod-eastus-rg', days=30{"resource_group": "resource_group='prod-eastus-rg', days=30", "period_days": 30, "total_cost_usd": 24867.71, "trend_vs_prior_period": "up 12%", "top_services": ["Azure Kubernetes Service", "Azure SQL MI", "Blob Storage"]}I have the current spend and trend data for the 'prod-eastus-rg' resource group. Next, I need to identify any orphaned resources that can be cleaned up to save costs.

Action: list_orphaned_resources
Action Input: resource_group='prod-eastus-rg'{"resource_group": "resource_group='prod-eastus-rg'", "orphaned_resources": [{"name": "vm-dev-unused-01", "type": "Virtual Machine", "idle_days": 45, "

/tmp/ipykernel_19437/3780442622.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_evaluated"      : datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")


I have the compliance posture data for the 'sub-prod-001' subscription. Next, I need to calculate the potential savings from using Reserved Instances (RIs) for 20 D4s_v3 VMs on a 1-year term.

Action: calculate_reserved_instance_savings
Action Input: input='{"vm_family": "D4s_v3", "quantity": 20, "term_years": 1}'{"vm_family": "D4s_v3", "quantity": 20, "term_years": 1, "payg_monthly_usd": 9411.9, "ri_monthly_usd": 6117.74, "discount_pct": "35%", "annual_savings_usd": 39529.98}Thought: I now have enough data to give a final answer

Final Answer: 
Cloud Optimization Report for 'prod-eastus-rg' in 'sub-prod-001':

1. Current Spend and Trend:
   - Total cost for the last 30 days: $24,867.71
   - Trend vs. prior period: Up 12%
   - Top services by cost: Azure Kubernetes Service, Azure SQL MI, Blob Storage

2. Orphaned Resources and Potential Savings:
   - Identified orphaned resources: 
     * vm-dev-unused-01 (Virtual Machine, idle for 45 days, monthly cost: $312)
     * disk-backup-old-02

### Step 10.4 — Enable MLflow Tracing for Agents

In [ ]:
import mlflow
import os

# ── Use SQLite backend (recommended over file-based mlruns) ───────────────
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("agent_experiments")
mlflow.langchain.autolog()                          # no extra args needed

with mlflow.start_run(run_name="finops_agent_run"):
    traced_result = agent_executor.invoke({
        "input": "Check costs and compliance for resource group prod-eastus-rg in subscription sub-prod-001"
    })

print("Agent run logged to MLflow")
print("   View: run 'mlflow ui --backend-store-uri sqlite:///mlflow.db' → open http://localhost:5000")
print()
print("RESULT:")
print(traced_result["output"])

2026/03/11 17:41:30 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x78d34ed62de0> at 0x78d32d8d8e00> was created in a different Context
2026/03/11 17:41:30 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x78d34ed62de0> at 0x78d32cf6fe40> was created in a different Context




> Entering new AgentExecutor chain...


2026/03/11 17:41:32 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x78d34ed62de0> at 0x78d32cfeff80> was created in a different Context
2026/03/11 17:41:32 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x78d34ed62de0> at 0x78d32d4245c0> was created in a different Context


To address the question, I need to gather data on both the costs associated with the specified resource group and the compliance status of the subscription it belongs to.

Action: get_azure_resource_cost
Action Input: resource_group="prod-eastus-rg", days=30{"resource_group": "resource_group=\"prod-eastus-rg\", days=30", "period_days": 30, "total_cost_usd": 37041.79, "trend_vs_prior_period": "stable", "top_services": ["Azure Kubernetes Service", "Azure SQL MI", "Blob Storage"]}

/tmp/ipykernel_19437/3780442622.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_evaluated"      : datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
2026/03/11 17:41:33 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x78d34ed62de0> at 0x78d32d118540> was created in a different Context
2026/03/11 17:41:33 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x78d34ed62de0> at 0x78d32d00f640> was created in a different Context


Now that I have the cost data for the prod-eastus-rg resource group, I need to check the compliance status of the subscription it belongs to, which is sub-prod-001.

Action: check_compliance_status
Action Input: subscription_id="sub-prod-001"{"subscription_id": "subscription_id=\"sub-prod-001", "compliance_score": "95%", "total_policies": 124, "violations": 9, "critical_violations": 1, "last_evaluated": "2026-03-11T17:41:33Z"}Thought: I now have enough data to give a final answer, including the costs for the prod-eastus-rg resource group and the compliance status of the sub-prod-001 subscription.

Final Answer: 
The prod-eastus-rg resource group in the sub-prod-001 subscription has incurred a total cost of $37,041.79 USD over the past 30 days, with a stable trend compared to the prior period. The top services contributing to this cost are Azure Kubernetes Service, Azure SQL MI, and Blob Storage.

Regarding compliance, the sub-prod-001 subscription has a compliance score of 95%, with 9 

---
# Lab Complete!

## What You Built

| Section | Pattern | Production Ready? |
|---------|---------|-------------------|
| 0 — Setup | Azure credentials, LLM factory, smoke test | ✅ Yes |
| 1 — Azure OpenAI | Direct invocation, JSON structured output | ✅ Yes |
| 2 — LCEL Chains | Summarization chain, batch processing | ✅ Yes |
| 3 — Sequential | Multi-step pipeline with state passing | ✅ Yes |
| 4 — Router | Domain classifier + specialized sub-chains | ✅ Yes |
| 5 — Memory | Stateful multi-turn chat with history trimming | ✅ Yes |
| 6 — Vector Store | FAISS index with Azure embeddings, persistence | ✅ Yes |
| 7 — RAG Chain | Retrieval-augmented generation with citation | ✅ Yes |
| 8 — MLflow | Experiment runs, metrics, model registry | ✅ Yes |
| 9 — Databricks RAG | Unity Catalog, Vector Search, production RAG | ✅ Yes |
| 10 — Agents | Custom tools, ReAct agent, MLflow tracing | ✅ Yes |

---

## Quick Reference

```python
# 1. BASIC CHAIN
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"key": "value"})

# 2. SEQUENTIAL WITH STATE
pipeline = (
    {"input": RunnablePassthrough()}
    | RunnablePassthrough.assign(step1=chain1)
    | RunnablePassthrough.assign(step2=chain2)
)

# 3. PARALLEL
parallel = RunnableParallel(a=chain_a, b=chain_b)
# Returns {'a': ..., 'b': ...}

# 4. CONDITIONAL ROUTING
router = RunnableBranch(
    (lambda x: condition, handler_chain),
    default_chain,
)

# 5. BATCH PROCESSING
results = chain.batch([{"k": v1}, {"k": v2}])

# 6. STREAMING
for chunk in chain.stream(input):
    print(chunk, end="", flush=True)

# 7. FALLBACK RESILIENCE
chain = primary.with_fallbacks([fallback])

# 8. MLFLOW TRACING
mlflow.langchain.autolog()
with mlflow.start_run():
    result = chain.invoke(input)

# 9. PYTHON TRANSFORMATION
step = RunnableLambda(my_python_function)

# 10. CALLBACKS
chain.invoke(input, config={"callbacks": [handler]})
```

---

## Recommended Next Steps

1. **Persist FAISS to Azure Blob Storage** — use `FAISS.save_local()` + Azure Storage SDK  
2. **Send metrics to Azure Application Insights** — integrate `opencensus-ext-azure` in the callback handler  
3. **Deploy as a REST API** — wrap your RAG chain in FastAPI + deploy to Azure Container Apps  
4. **Add LangSmith tracing** — full pipeline observability across production chains  
5. **Replace FAISS with Azure AI Search** — managed, scalable hybrid (keyword + semantic) search  
6. **Version your prompts in MLflow** — log prompt templates as params to compare iterations  
